# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLWithMathematics/Fly_Rank_ML-Intern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item (page), identified by content_id. There are 30,000 rows × 44 columns across 32 pseudonymized clients.

Time window: All numeric metrics are aggregated over a trailing 90-day window ending at the export date. The 30-day comparison sub-windows (impressions_last_30d, impressions_prev_30d, etc.) split this 90-day window into a most-recent 30 days and a prior 30 days (days 31–60 back). There is no explicit report_date column — the entire dataset is a single snapshot.

For clustering: Since this is unsupervised, there is no separate "feature window" vs "target window." All signals come from the same 90-day trailing snapshot. The cluster assignment itself is the output, not a prediction about the future.

In [6]:

import pandas as pd
import numpy as np
import os
LOCAL_PATH = "data/raw/content_refresh_anonymized.csv"
RAW_URL = "https://raw.githubusercontent.com/MLWithMathematics/Fly_Rank_ML-Intern/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(LOCAL_PATH if os.path.exists(LOCAL_PATH) else RAW_URL)

# --- Grain check: one row per content_id ---
total_rows = len(df)
unique_ids = df['content_id'].nunique()
print(f"Total rows:         {total_rows:,}")
print(f"Unique content_id:  {unique_ids:,}")
print(f"Grain is 1:1:       {total_rows == unique_ids}")

# --- Grain probe: any duplicates? ---
dupes = df.groupby('content_id').size().reset_index(name='cnt')
dupes = dupes[dupes['cnt'] > 1]
print(f"\nDuplicate content_ids: {len(dupes)}")  # expect 0

# --- Shape and clients ---
print(f"\nShape: {df.shape}")
print(f"Clients: {df['client_id'].nunique()}")

# --- Time window verification (no report_date, but age confirms 90-day minimum) ---
print(f"\ncontent_age_days — min: {df['content_age_days'].min()}, "
      f"max: {df['content_age_days'].max()}")
print(f"days_with_impressions — min: {df['days_with_impressions'].min()}, "
      f"max: {df['days_with_impressions'].max()}")

Total rows:         30,000
Unique content_id:  30,000
Grain is 1:1:       True

Duplicate content_ids: 0

Shape: (30000, 44)
Clients: 32

content_age_days — min: 90, max: 564
days_with_impressions — min: 1, max: 88


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Excluded (with reasons)
Column	Why excluded
impressions_last_30d	Raw input to trend_pct → using it leaks the label direction
impressions_prev_30d	Same — raw input to trend_pct
clicks_last_30d	Sub-window of 90d; correlated with trend direction
clicks_prev_30d	Same
sessions_last_30d	Sub-window of 90d; correlated with trend direction
sessions_prev_30d	Same
provider_used	LLM provider; not a content performance signal, not a model feature per data dictionary
model_used	LLM model name; same rationale — not a model feature

In [7]:

# --- Field classification verification ---
FEATURES_NUMERIC = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'content_age_days', 'days_since_last_update',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d',
    'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
    'age_tier_order'
]
FEATURES_CATEGORICAL = [
    'content_type', 'main_intent', 'competition_level',
    'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier',
    'impression_tier', 'position_tier'
]

LABEL_PROXY = ['trend_direction', 'trend_pct']
# is_declining_label only exists after prep step, not in raw CSV

CONTEXT = ['content_id', 'client_id']

EXCLUDED = [
    'impressions_last_30d', 'impressions_prev_30d',
    'clicks_last_30d', 'clicks_prev_30d',
    'sessions_last_30d', 'sessions_prev_30d',
    'provider_used', 'model_used'
]

# Verify all 44 columns are accounted for
all_classified = set(FEATURES_NUMERIC + FEATURES_CATEGORICAL +
                     LABEL_PROXY + CONTEXT + EXCLUDED)
all_columns = set(df.columns)

unclassified = all_columns - all_classified
extra = all_classified - all_columns
print(f"Total columns in CSV:     {len(all_columns)}")
print(f"Total columns classified: {len(all_classified)}")
print(f"Unclassified columns:     {unclassified if unclassified else '✓ None'}")
print(f"Extra (not in CSV):       {extra if extra else '✓ None'}")

# Verify no leakage columns in feature sets
leakage_set = set(LABEL_PROXY + EXCLUDED + CONTEXT)
feature_set = set(FEATURES_NUMERIC + FEATURES_CATEGORICAL)
leak_check = leakage_set & feature_set
print(f"\nLeakage check:            {'✓ Clean' if not leak_check else f'⚠ LEAK: {leak_check}'}")

Total columns in CSV:     44
Total columns classified: 44
Unclassified columns:     ✓ None
Extra (not in CSV):       ✓ None

Leakage check:            ✓ Clean


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:

# ============================================================
# 3a. GRAIN VERIFICATION
# ============================================================
print("=" * 60)
print("3a. GRAIN CHECK")
print("=" * 60)

# The grain is content_id — should be 1:1
grain_check = df.groupby('content_id').size()
dupes = grain_check[grain_check > 1]
print(f"Rows with duplicate content_id: {len(dupes)}")
print(f"→ {'✓ Grain holds: one row per content_id' if len(dupes) == 0 else '⚠ GRAIN BROKEN'}")

# ============================================================
# 3b. ROW COUNTS
# ============================================================
print("\n" + "=" * 60)
print("3b. ROW COUNTS")
print("=" * 60)
print(f"Total rows:       {len(df):,}  (expected: 30,000)")
print(f"Unique pages:     {df['content_id'].nunique():,}")
print(f"Unique clients:   {df['client_id'].nunique():,}  (expected: 32)")
print(f"Content types:    {df['content_type'].value_counts().to_dict()}")

# ============================================================
# 3c. MISSINGNESS — overall + by content_type
# ============================================================
print("\n" + "=" * 60)
print("3c. MISSINGNESS — overall")
print("=" * 60)
miss_pct = (df.isnull().sum() / len(df) * 100).round(2)
miss_nonzero = miss_pct[miss_pct > 0].sort_values(ascending=False)
print(miss_nonzero.to_string())

print("\n" + "=" * 60)
print("3c. MISSINGNESS — by content_type (patterned gaps)")
print("=" * 60)
key_cols = ['search_volume', 'competition', 'competition_level', 'cpc',
            'main_intent', 'word_count', 'char_count']
for ct in df['content_type'].unique():
    subset = df[df['content_type'] == ct]
    print(f"\n  content_type = '{ct}' ({len(subset):,} rows)")
    for col in key_cols:
        miss = subset[col].isna().mean() * 100
        if miss > 0:
            print(f"    {col}: {miss:.1f}% missing")

# ============================================================
# 3d. WINDOW VERIFICATION
# ============================================================
print("\n" + "=" * 60)
print("3d. WINDOW / RANGE CHECKS")
print("=" * 60)

# content_age_days — all rows should be >= 90
print(f"content_age_days — min: {df['content_age_days'].min()}, "
      f"max: {df['content_age_days'].max()}")
print(f"  Rows with age < 90: {(df['content_age_days'] < 90).sum()}")

# impressions_90d — all rows >= 1
print(f"impressions_90d — min: {df['impressions_90d'].min()}, "
      f"max: {df['impressions_90d'].max():,}")
print(f"  Rows with 0 impressions: {(df['impressions_90d'] == 0).sum()}")

# avg_position = 0 means "no data"
print(f"avg_position = 0 (no data): {(df['avg_position'] == 0).sum():,} rows")

# days_with_impressions range
print(f"days_with_impressions — min: {df['days_with_impressions'].min()}, "
      f"max: {df['days_with_impressions'].max()}")

# ============================================================
# 3e. RATE COLUMN SCALE CHECK
# ============================================================
print("\n" + "=" * 60)
print("3e. RATE COLUMNS — ×100 scale verification")
print("=" * 60)
for col in ['ctr', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']:
    vals = df[col].dropna()
    print(f"{col:20s} — median: {vals.median():.2f}, "
          f"max: {vals.max():.2f}, >100: {(vals > 100).sum()}")

# ============================================================
# 3f. LEAKAGE AUDIT
# ============================================================
print("\n" + "=" * 60)
print("3f. LEAKAGE AUDIT")
print("=" * 60)
# trend_direction distribution — this is the label source
print("trend_direction distribution:")
print(df['trend_direction'].value_counts().to_string())
print(f"\ntrend_pct — min: {df['trend_pct'].min():.1f}, "
      f"max: {df['trend_pct'].max():.1f}, "
      f"NaN: {df['trend_pct'].isna().sum()}")
# Verify trend_pct and 30-day sub-windows are NOT in feature list
for col in LABEL_PROXY + EXCLUDED[:6]:
    print(f"  '{col}' in features? "
          f"{'⚠ YES — LEAK!' if col in FEATURES_NUMERIC + FEATURES_CATEGORICAL else '✓ No'}")

3a. GRAIN CHECK
Rows with duplicate content_id: 0
→ ✓ Grain holds: one row per content_id

3b. ROW COUNTS
Total rows:       30,000  (expected: 30,000)
Unique pages:     30,000
Unique clients:   32  (expected: 32)
Content types:    {'keyword article': 27207, 'feedly article': 2096, 'comparison article': 697}

3c. MISSINGNESS — overall
provider_used        71.46
word_count_tier      25.66
char_count           25.66
word_count           25.66
char_count_tier      25.66
model_used           19.11
trend_pct            11.29
competition_level     8.70
search_volume         8.23
competition           8.23
cpc                   8.23
main_intent           7.91
scroll_rate           0.42

3c. MISSINGNESS — by content_type (patterned gaps)

  content_type = 'keyword article' (27,207 rows)
    search_volume: 1.4% missing
    competition: 1.4% missing
    competition_level: 1.9% missing
    cpc: 1.4% missing
    main_intent: 1.0% missing
    word_count: 28.3% missing
    char_count: 28.3% missing



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

1. No future outcomes. The starter dataset is a single 90-day snapshot — there is no "next month" to observe. Clustering discovers current performance archetypes, not predictions about what will happen. Any claim about "these pages will decline" requires the warehouse release with proper future-window construction.

2. Missingness follows content_type, not random. feedly article rows have ~100% missing keyword data (search_volume, competition, cpc, competition_level). comparison article rows have ~28% missing word_count. A blind fillna(0) silently encodes content type into numeric features. Mitigation: add has_keyword_data, has_word_count flags before imputing.

3. avg_position = 0 is "no data," not rank zero. 1,205 rows have avg_position = 0. Including them raw pushes the mean position down. Mitigation: recode to NaN or add a has_position flag.

4. Rate columns can exceed 100. scroll_rate and ai_traffic_pct use different numerator/denominator measurement systems. Values > 100 are real, not errors — but they can distort standardized feature scaling if not expected.

5. No causal claims possible. Clustering shows what exists, not what works. A cluster called "stale visible page" does not prove that refreshing those pages will recover traffic — that would require an experiment.

6. Unbalanced panel (warehouse only, but worth noting). Different clients have different history depths. When scaling to the warehouse, per-client date checks via dim_clients.gsc_data_start are mandatory before defining any time window.

In [9]:
# --- Demonstrate the key data limits ---

# 4a. Missingness is patterned, not random
print("4a. Keyword missingness by content_type:")
for ct in df['content_type'].unique():
    subset = df[df['content_type'] == ct]
    sv_miss = subset['search_volume'].isna().mean() * 100
    wc_miss = subset['word_count'].isna().mean() * 100
    print(f"  {ct:25s} — search_volume missing: {sv_miss:.1f}%, "
          f"word_count missing: {wc_miss:.1f}%")

# 4b. avg_position = 0 trap
print(f"\n4b. avg_position = 0 (no data): {(df['avg_position'] == 0).sum():,} / "
      f"{len(df):,} rows")
pos_zero = df[df['avg_position'] == 0]
print(f"    Those rows' median impressions_90d: "
      f"{pos_zero['impressions_90d'].median():.0f}")
print(f"    → Low-impression pages without position data. "
      f"Not 'rank zero' — treat as missing.")

# 4c. Rates exceeding 100
print(f"\n4c. scroll_rate > 100: {(df['scroll_rate'] > 100).sum()} rows")
print(f"    ai_traffic_pct > 100: {(df['ai_traffic_pct'] > 100).sum()} rows")
print(f"    → Not errors; different measurement systems. "
      f"Handle in scaling, don't clip.")

# 4d. Starter slice scope
print(f"\n4d. Starter slice scope:")
print(f"    Rows: {len(df):,} (vs ~519k content items in full warehouse)")
print(f"    Clients: {df['client_id'].nunique()} (vs 104 in full warehouse)")
print(f"    → Results are 'observed in the 30k-row starter slice', "
      f"not 'proven at scale'")

# 4e. Show that fillna(0) on search_volume encodes content_type
print(f"\n4e. Demonstrating fillna(0) danger:")
print(f"    Correlation of search_volume==0 with content_type (after fillna):")
df_test = df.copy()
df_test['sv_zero'] = df_test['search_volume'].fillna(0) == 0
for ct in df_test['content_type'].unique():
    pct = df_test[df_test['content_type'] == ct]['sv_zero'].mean() * 100
    print(f"      {ct:25s}: {pct:.1f}% have search_volume=0 after fillna")
print(f"    → fillna(0) creates a near-perfect proxy for content_type. "
      f"Use has_keyword_data flag instead.")

4a. Keyword missingness by content_type:
  keyword article           — search_volume missing: 1.4%, word_count missing: 28.3%
  feedly article            — search_volume missing: 100.0%, word_count missing: 0.0%
  comparison article        — search_volume missing: 0.0%, word_count missing: 0.0%

4b. avg_position = 0 (no data): 1,205 / 30,000 rows
    Those rows' median impressions_90d: 1
    → Low-impression pages without position data. Not 'rank zero' — treat as missing.

4c. scroll_rate > 100: 119 rows
    ai_traffic_pct > 100: 23 rows
    → Not errors; different measurement systems. Handle in scaling, don't clip.

4d. Starter slice scope:
    Rows: 30,000 (vs ~519k content items in full warehouse)
    Clients: 32 (vs 104 in full warehouse)
    → Results are 'observed in the 30k-row starter slice', not 'proven at scale'

4e. Demonstrating fillna(0) danger:
    Correlation of search_volume==0 with content_type (after fillna):
      keyword article          : 39.5% have search_volume=0

## Self-check

Before you submit, confirm each line honestly:

- [ ✅ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ✅ ] No client names, URLs, or private queries anywhere
- [ ✅ ] My claims use careful words: observed, measured, directional, decision-support
- [ ✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.